In [1]:
# === TRAJECTORY CLUSTERING – MEDITERRANEAN PILOT ===
# 
# RQ3: Lassen sich distinkte Erholungstypen (Cluster) identifizieren,
#       und wie korrelieren diese mit Ökoregion und Landbedeckung?
#
# Workflow:
#   1. Daten laden & auf Mediterran filtern
#   2. Feature-Extraktion pro Pixel (8 Features aus Trajektorie)
#   3. Standardisierung & NaN-Handling
#   4. SOM-Training (Dimensionsreduktion)
#   5. k-Medoids auf SOM-Codebook (Cluster-Bestimmung)
#   6. Cluster-Validierung (Silhouette, Calinski-Harabasz, Davies-Bouldin)
#   7. Cluster-Interpretation (Trajektorien, LC-Zusammensetzung)
#   8. Räumliche Karte & Export

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
import os
import time
from tqdm import tqdm
from scipy import stats

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

print("=" * 70)
print("TRAJECTORY CLUSTERING – MEDITERRANEAN PILOT")
print("=" * 70)

TRAJECTORY CLUSTERING – MEDITERRANEAN PILOT


In [2]:
# === SCHRITT 0: KONFIGURATION & DATEN LADEN ===

print("\n0. KONFIGURATION & DATEN LADEN")
print("-" * 70)

# === Band-Konfiguration (identisch zu 05_landcover_analysis.ipynb) ===
years_woody = list(range(1985, 2025))
years_burned = list(range(2000, 2026))
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))

WOODY_START = 1
MBA_START = len(years_woody) + 1
MODIS_START = len(years_woody) + len(years_burned) * 12 + 1

# === Pfade ===
combined_path = os.path.join(
    workDir, r"_Runs\05_Landcover_integrated\woody_burned_MBA_MODIS_IGBP_combined.tif"
)
ecoregion_raster_path = os.path.join(
    workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_500m_3035_MBA.tif"
)
ecoregion_attr_path = os.path.join(
    workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_attributes_with_colors.csv"
)

out_dir = os.path.join(workDir, "_Runs", "06_Clustering")
os.makedirs(out_dir, exist_ok=True)

# === Lade Daten ===
print("Lade Woody Cover...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    woody_bands = list(range(WOODY_START, WOODY_START + len(years_woody)))
    woody = src.read(woody_bands)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width
    meta = src.meta.copy()
print(f"  ✓ Woody Cover: {woody.shape} ({time.time()-t0:.1f}s)")

print("Lade Burned Area...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    burned_band_indices = [MBA_START + (i * 12) for i in range(len(years_burned))]
    burned = src.read(burned_band_indices)
print(f"  ✓ Burned Area: {burned.shape} ({time.time()-t0:.1f}s)")

print("Lade MODIS IGBP...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    modis_bands = list(range(MODIS_START, MODIS_START + len(modis_years)))
    modis_igbp = src.read(modis_bands)
print(f"  ✓ MODIS IGBP: {modis_igbp.shape} ({time.time()-t0:.1f}s)")

print("Lade Ecoregion-Raster...")
with rasterio.open(ecoregion_raster_path) as src:
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)
eco_mapping = {}
for _, row in ecoregion_df.iterrows():
    eco_mapping[row['NUMERIC_ID']] = {
        'code': row['code'], 'name': row['name'], 'hex_color': row['hex_color']
    }

# === Vorberechnungen ===
fire_counts = np.sum(burned == 1, axis=0)
first_fire_idx = np.argmax(burned == 1, axis=0)

print(f"\n✓ Alle Daten geladen!")# === SCHRITT 0: KONFIGURATION & DATEN LADEN ===

print("\n0. KONFIGURATION & DATEN LADEN")
print("-" * 70)

# === Band-Konfiguration (identisch zu 05_landcover_analysis.ipynb) ===
years_woody = list(range(1985, 2025))
years_burned = list(range(2000, 2026))
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))

WOODY_START = 1
MBA_START = len(years_woody) + 1
MODIS_START = len(years_woody) + len(years_burned) * 12 + 1

# === Pfade ===
combined_path = os.path.join(
    workDir, r"_Runs\05_Landcover_integrated\woody_burned_MBA_MODIS_IGBP_combined.tif"
)
ecoregion_raster_path = os.path.join(
    workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_500m_3035_MBA.tif"
)
ecoregion_attr_path = os.path.join(
    workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_attributes_with_colors.csv"
)

out_dir = os.path.join(workDir, "_Runs", "06_Clustering")
os.makedirs(out_dir, exist_ok=True)

# === Lade Daten ===
print("Lade Woody Cover...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    woody_bands = list(range(WOODY_START, WOODY_START + len(years_woody)))
    woody = src.read(woody_bands)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width
    meta = src.meta.copy()
print(f"  ✓ Woody Cover: {woody.shape} ({time.time()-t0:.1f}s)")

print("Lade Burned Area...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    burned_band_indices = [MBA_START + (i * 12) for i in range(len(years_burned))]
    burned = src.read(burned_band_indices)
print(f"  ✓ Burned Area: {burned.shape} ({time.time()-t0:.1f}s)")

print("Lade MODIS IGBP...")
t0 = time.time()
with rasterio.open(combined_path) as src:
    modis_bands = list(range(MODIS_START, MODIS_START + len(modis_years)))
    modis_igbp = src.read(modis_bands)
print(f"  ✓ MODIS IGBP: {modis_igbp.shape} ({time.time()-t0:.1f}s)")

print("Lade Ecoregion-Raster...")
with rasterio.open(ecoregion_raster_path) as src:
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)
eco_mapping = {}
for _, row in ecoregion_df.iterrows():
    eco_mapping[row['NUMERIC_ID']] = {
        'code': row['code'], 'name': row['name'], 'hex_color': row['hex_color']
    }

# === Vorberechnungen ===
fire_counts = np.sum(burned == 1, axis=0)
first_fire_idx = np.argmax(burned == 1, axis=0)

print(f"\n✓ Alle Daten geladen!")


0. KONFIGURATION & DATEN LADEN
----------------------------------------------------------------------
Lade Woody Cover...
  ✓ Woody Cover: (40, 9660, 10483) (83.1s)
Lade Burned Area...
  ✓ Burned Area: (26, 9660, 10483) (65.7s)
Lade MODIS IGBP...
  ✓ MODIS IGBP: (24, 9660, 10483) (53.8s)
Lade Ecoregion-Raster...

✓ Alle Daten geladen!

0. KONFIGURATION & DATEN LADEN
----------------------------------------------------------------------
Lade Woody Cover...
  ✓ Woody Cover: (40, 9660, 10483) (71.0s)
Lade Burned Area...
  ✓ Burned Area: (26, 9660, 10483) (61.8s)
Lade MODIS IGBP...
  ✓ MODIS IGBP: (24, 9660, 10483) (54.2s)
Lade Ecoregion-Raster...

✓ Alle Daten geladen!


In [ ]:
# === SCHRITT 1: MEDITERRAN-MASKE & PRE-FIRE LANDCOVER ===

print("\n1. MEDITERRAN-MASKE & PRE-FIRE LANDCOVER")
print("-" * 70)

# Finde Mediterranean Ecoregion ID
MED_ID = None
for eco_id, info in eco_mapping.items():
    if info['code'] == 'MED':
        MED_ID = eco_id
        break

if MED_ID is None:
    # Fallback: Suche nach "Mediterr" im Namen
    for eco_id, info in eco_mapping.items():
        if 'mediterr' in info['name'].lower():
            MED_ID = eco_id
            break

print(f"  Mediterranean ID: {MED_ID}")
print(f"  Name: {eco_mapping[MED_ID]['name']}")

# Erstelle Maske: Mediterran + genau 1 Brand
med_mask = (eco_raster == MED_ID)
med_single_fire_mask = med_mask & (fire_counts == 1)

n_med_total = np.sum(med_mask)
n_med_burned = np.sum(med_mask & (fire_counts > 0))
n_med_single = np.sum(med_single_fire_mask)

print(f"\n  Pixel in Mediterran gesamt: {n_med_total:,}")
print(f"  Pixel mit ≥1 Brand:         {n_med_burned:,}")
print(f"  Pixel mit genau 1 Brand:     {n_med_single:,}")

# === Pre-Fire Landcover (wie in 05_landcover_analysis) ===
MAJOR_LC_NAMES = ['', 'Forests', 'Shrublands', 'Savannas', 'Grasslands',
                  'Wetlands', 'Croplands', 'Urban', 'Barren/Ice', 'Water', 'Other']
igbp_to_major_id = np.array([
    0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 6, 8, 8, 9
], dtype=np.uint8)

fire_year_values = np.array(years_burned)[first_fire_idx]
target_years = fire_year_values - 1
modis_idx = np.clip(target_years - modis_years[0], 0, len(modis_years) - 1)

rows_fire, cols_fire = np.where(med_single_fire_mask)

pre_fire_lc = np.zeros((height, width), dtype=np.uint8)
pre_fire_lc[rows_fire, cols_fire] = modis_igbp[
    modis_idx[rows_fire, cols_fire], rows_fire, cols_fire
]
invalid = (pre_fire_lc < 1) | (pre_fire_lc > 17)
pre_fire_lc[invalid] = 0

pre_fire_lc_major_id = igbp_to_major_id[np.clip(pre_fire_lc, 0, 17)]

# LC Verteilung in Mediterran
unique_lc_med, counts_lc_med = np.unique(
    pre_fire_lc_major_id[med_single_fire_mask & (pre_fire_lc > 0)],
    return_counts=True
)
print(f"\n  LC-Verteilung in Mediterran (Single Fire):")
for mid, cnt in zip(unique_lc_med, counts_lc_med):
    pct = cnt / counts_lc_med.sum() * 100
    print(f"    {MAJOR_LC_NAMES[mid]:20s}: {cnt:>8,} ({pct:.1f}%)")

In [ ]:
# === SCHRITT 2: FEATURE-EXTRAKTION PRO PIXEL ===

print("\n2. FEATURE-EXTRAKTION PRO PIXEL")
print("-" * 70)

# Pixel-Koordinaten für Mediterran mit 1 Brand
y_med, x_med = np.where(med_single_fire_mask)
n_pixels = len(y_med)
print(f"  Extrahiere Features für {n_pixels:,} Pixel...")

offset = years_burned[0] - years_woody[0]  # 15
fire_idx_med = first_fire_idx[y_med, x_med]

# === Extrahiere Woody-Cover-Werte für relative Jahre -5 bis +10 ===
rel_years_range = list(range(-5, 11))  # 16 Zeitschritte
pixel_trajectories = np.full((n_pixels, len(rel_years_range)), np.nan, dtype=np.float32)

print("  Extrahiere Trajektorien...")
for j, rel_year in enumerate(tqdm(rel_years_range, desc="Rel Years")):
    woody_band_idx = fire_idx_med + rel_year + offset - 1
    valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))
    
    if np.sum(valid) > 0:
        vals = woody[woody_band_idx[valid], y_med[valid], x_med[valid]].astype(np.float32)
        vals[vals == nodata] = np.nan
        pixel_trajectories[valid, j] = vals

print(f"  ✓ Trajektorien Shape: {pixel_trajectories.shape}")
print(f"  NaN-Anteil: {np.isnan(pixel_trajectories).mean()*100:.1f}%")

# === Berechne 8 Features pro Pixel ===
print("\n  Berechne Features...")

# Indices in pixel_trajectories:
# -5=0, -4=1, -3=2, -2=3, -1=4, 0=5, 1=6, 2=7, 3=8, 4=9, 5=10, ..., 10=15
pre_fire = pixel_trajectories[:, 0:5]     # -5 bis -1
fire_year = pixel_trajectories[:, 5]       # 0
year_m1 = pixel_trajectories[:, 4]         # -1
year_p1 = pixel_trajectories[:, 6]         # +1
year_p3 = pixel_trajectories[:, 8]         # +3
year_p5 = pixel_trajectories[:, 10]        # +5

# Feature 1: Pre-fire mean woody cover
feat_pre_fire_mean = np.nanmean(pre_fire, axis=1)

# Feature 2: Pre-fire trend (slope)
feat_pre_fire_trend = np.full(n_pixels, np.nan, dtype=np.float32)
x_trend = np.arange(5, dtype=np.float32)
for i in range(n_pixels):
    vals_i = pre_fire[i, :]
    valid_i = ~np.isnan(vals_i)
    if np.sum(valid_i) >= 3:
        slope, _, _, _, _ = stats.linregress(x_trend[valid_i], vals_i[valid_i])
        feat_pre_fire_trend[i] = slope

# Feature 3: Absolute fire loss
feat_fire_loss = year_m1 - fire_year

# Feature 4: Relative fire loss (%)
feat_fire_loss_pct = np.where(year_m1 > 0, feat_fire_loss / year_m1 * 100, np.nan)

# Feature 5: Recovery after 1 year (absolute change from fire year)
feat_recovery_1yr = year_p1 - fire_year

# Feature 6: Recovery after 3 years
feat_recovery_3yr = year_p3 - fire_year

# Feature 7: Recovery after 5 years
feat_recovery_5yr = year_p5 - fire_year

# Feature 8: Recovery rate 5yr (% of loss recovered)
feat_recovery_rate_5yr = np.where(
    feat_fire_loss > 0,
    feat_recovery_5yr / feat_fire_loss * 100,
    np.nan
)

# === Kombiniere zu Feature-Matrix ===
FEATURE_NAMES = [
    'pre_fire_mean',
    'pre_fire_trend',
    'fire_loss',
    'fire_loss_pct',
    'recovery_1yr',
    'recovery_3yr',
    'recovery_5yr',
    'recovery_rate_5yr'
]

feature_matrix = np.column_stack([
    feat_pre_fire_mean,
    feat_pre_fire_trend,
    feat_fire_loss,
    feat_fire_loss_pct,
    feat_recovery_1yr,
    feat_recovery_3yr,
    feat_recovery_5yr,
    feat_recovery_rate_5yr
])

print(f"\n  Feature-Matrix Shape: {feature_matrix.shape}")
print(f"  NaN pro Feature:")
for i, name in enumerate(FEATURE_NAMES):
    nan_pct = np.isnan(feature_matrix[:, i]).mean() * 100
    print(f"    {name:25s}: {nan_pct:.1f}% NaN")

# === Entferne Pixel mit zu vielen NaNs ===
nan_count_per_pixel = np.isnan(feature_matrix).sum(axis=1)
valid_pixels = nan_count_per_pixel == 0  # Nur Pixel OHNE NaN behalten

feature_matrix_clean = feature_matrix[valid_pixels]
y_valid = y_med[valid_pixels]
x_valid = x_med[valid_pixels]
trajectories_clean = pixel_trajectories[valid_pixels]
lc_valid = pre_fire_lc_major_id[y_valid, x_valid]

n_valid = feature_matrix_clean.shape[0]
print(f"\n  ✓ Gültige Pixel (kein NaN): {n_valid:,} ({n_valid/n_pixels*100:.1f}%)")
print(f"  Entfernt: {n_pixels - n_valid:,} Pixel")

In [ ]:
# === SCHRITT 3: STANDARDISIERUNG ===

print("\n3. STANDARDISIERUNG")
print("-" * 70)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
feature_matrix_scaled = scaler.fit_transform(feature_matrix_clean)

print(f"  Shape: {feature_matrix_scaled.shape}")
print(f"  Feature-Statistik nach Standardisierung:")
for i, name in enumerate(FEATURE_NAMES):
    print(f"    {name:25s}: mean={feature_matrix_scaled[:,i].mean():.4f}, "
          f"std={feature_matrix_scaled[:,i].std():.4f}")

In [ ]:
# === SCHRITT 4: SOM-TRAINING ===

print("\n4. SOM-TRAINING (Self-Organizing Map)")
print("-" * 70)

from minisom import MiniSom

# SOM-Größe: Heuristik ≈ 5 * sqrt(n_samples), aber max sinnvolle Größe
som_side = min(30, int(np.ceil(5 * np.sqrt(n_valid) ** 0.5)))
# Für sehr große Datensätze auf 20×20 oder 25×25 begrenzen
som_side = max(10, min(som_side, 25))

print(f"  SOM-Größe: {som_side} × {som_side} = {som_side**2} Neuronen")
print(f"  Input-Dimension: {feature_matrix_scaled.shape[1]} Features")
print(f"  Training-Samples: {n_valid:,}")

# Subsampling für schnelleres Training falls >200k Pixel
MAX_SOM_SAMPLES = 200000
if n_valid > MAX_SOM_SAMPLES:
    rng = np.random.default_rng(42)
    som_sample_idx = rng.choice(n_valid, MAX_SOM_SAMPLES, replace=False)
    som_training_data = feature_matrix_scaled[som_sample_idx]
    print(f"  ⚠️  Subsampled zu {MAX_SOM_SAMPLES:,} für SOM-Training")
else:
    som_training_data = feature_matrix_scaled
    som_sample_idx = None

# === Training ===
som = MiniSom(
    som_side, som_side,
    feature_matrix_scaled.shape[1],
    sigma=max(1.0, som_side / 4),
    learning_rate=0.5,
    neighborhood_function='gaussian',
    random_seed=42
)

print("  Initialisiere mit PCA...")
som.pca_weights_init(som_training_data)

n_iterations = 50000
print(f"  Trainiere SOM ({n_iterations:,} Iterationen)...")
t0 = time.time()
som.train_random(som_training_data, n_iterations, verbose=False)
print(f"  ✓ SOM-Training abgeschlossen ({time.time()-t0:.1f}s)")

# Quantisierungsfehler
qe = som.quantization_error(som_training_data)
print(f"  Quantisierungsfehler: {qe:.4f}")

# === Mapping: Jedes Pixel → bester SOM-Knoten ===
print("  Weise alle Pixel SOM-Knoten zu...")
t0 = time.time()
bmu_indices = np.array([som.winner(x) for x in tqdm(feature_matrix_scaled, desc="BMU")])
print(f"  ✓ BMU-Zuweisung abgeschlossen ({time.time()-t0:.1f}s)")

# Codebook extrahieren
codebook = som.get_weights().reshape(-1, feature_matrix_scaled.shape[1])
print(f"  Codebook Shape: {codebook.shape}")

In [ ]:
# === SCHRITT 5: k-MEDOIDS CLUSTERING AUF SOM-CODEBOOK ===

print("\n5. k-MEDOIDS CLUSTERING")
print("-" * 70)

from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# === Cluster-Validierung für k = 2 bis 12 ===
K_RANGE = range(2, 13)

validation_results = []

print("  Teste k-Werte...")
for k in tqdm(K_RANGE, desc="k-Medoids"):
    kmed = KMedoids(n_clusters=k, metric='euclidean', random_state=42, max_iter=300)
    codebook_labels = kmed.fit_predict(codebook)
    
    # Weise Pixel-Labels via BMU zu
    bmu_flat = bmu_indices[:, 0] * som_side + bmu_indices[:, 1]
    pixel_labels = codebook_labels[bmu_flat]
    
    # Subsample für schnellere Metrik-Berechnung
    if n_valid > 50000:
        rng = np.random.default_rng(42)
        eval_idx = rng.choice(n_valid, 50000, replace=False)
    else:
        eval_idx = np.arange(n_valid)
    
    sil = silhouette_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx],
                           sample_size=min(10000, len(eval_idx)))
    ch = calinski_harabasz_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx])
    db = davies_bouldin_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx])
    
    validation_results.append({
        'k': k,
        'silhouette': sil,
        'calinski_harabasz': ch,
        'davies_bouldin': db,
        'inertia': kmed.inertia_
    })
    
    print(f"    k={k:2d}: Silhouette={sil:.3f}, CH={ch:.0f}, DB={db:.3f}")

df_validation = pd.DataFrame(validation_results)

# === Visualisierung der Validierungsmetriken ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(df_validation['k'], df_validation['silhouette'], 'bo-', linewidth=2)
ax.set_xlabel('k')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score (higher = better)')
ax.grid(True, alpha=0.3)
best_sil_k = df_validation.loc[df_validation['silhouette'].idxmax(), 'k']
ax.axvline(x=best_sil_k, color='red', linestyle='--', alpha=0.7, label=f'Best: k={best_sil_k}')
ax.legend()

ax = axes[0, 1]
ax.plot(df_validation['k'], df_validation['calinski_harabasz'], 'go-', linewidth=2)
ax.set_xlabel('k')
ax.set_ylabel('Calinski-Harabasz Index')
ax.set_title('Calinski-Harabasz (higher = better)')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(df_validation['k'], df_validation['davies_bouldin'], 'ro-', linewidth=2)
ax.set_xlabel('k')
ax.set_ylabel('Davies-Bouldin Index')
ax.set_title('Davies-Bouldin (lower = better)')
ax.grid(True, alpha=0.3)
best_db_k = df_validation.loc[df_validation['davies_bouldin'].idxmin(), 'k']
ax.axvline(x=best_db_k, color='blue', linestyle='--', alpha=0.7, label=f'Best: k={best_db_k}')
ax.legend()

ax = axes[1, 1]
ax.plot(df_validation['k'], df_validation['inertia'], 'mo-', linewidth=2)
ax.set_xlabel('k')
ax.set_ylabel('Inertia')
ax.set_title('Inertia / Elbow (lower = better)')
ax.grid(True, alpha=0.3)

plt.suptitle('Cluster Validation Metrics – Mediterranean Pilot', fontsize=14, fontweight='bold')
plt.tight_layout()
val_plot_path = os.path.join(out_dir, "cluster_validation_metrics.png")
plt.savefig(val_plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n  ✓ {val_plot_path}")

# Speichere Validierungsergebnisse
df_validation.to_csv(os.path.join(out_dir, "cluster_validation_results.csv"), index=False)

print(f"\n  Empfohlene k-Werte:")
print(f"    Best Silhouette:     k = {best_sil_k}")
print(f"    Best Davies-Bouldin: k = {best_db_k}")
print(f"\n  ⚠️  Wähle k im nächsten Schritt manuell basierend auf den Metriken!")

In [ ]:
# === SCHRITT 6: FINALE CLUSTERUNG MIT GEWÄHLTEM k ===

print("\n6. FINALE CLUSTERUNG")
print("-" * 70)

# ============================================
# ⚠️  HIER k MANUELL SETZEN nach Inspektion der Validierungsmetriken!
# ============================================
CHOSEN_K = 5  # <-- ANPASSEN nach Schritt 5
# ============================================

print(f"  Gewähltes k = {CHOSEN_K}")

kmed_final = KMedoids(n_clusters=CHOSEN_K, metric='euclidean', random_state=42, max_iter=300)
codebook_labels_final = kmed_final.fit_predict(codebook)

# Pixel-Labels via BMU
bmu_flat = bmu_indices[:, 0] * som_side + bmu_indices[:, 1]
cluster_labels = codebook_labels_final[bmu_flat]

# Cluster-Größen
unique_clusters, cluster_counts = np.unique(cluster_labels, return_counts=True)
print(f"\n  Cluster-Verteilung:")
for cl, cnt in zip(unique_clusters, cluster_counts):
    print(f"    Cluster {cl}: {cnt:,} Pixel ({cnt/n_valid*100:.1f}%)")

In [ ]:
# === SCHRITT 7: CLUSTER-INTERPRETATION ===

print("\n7. CLUSTER-INTERPRETATION")
print("-" * 70)

# =====================================================
# 7a: Mittlere Trajektorien pro Cluster
# =====================================================

print("\n7a. Mittlere Trajektorien pro Cluster...")

CLUSTER_COLORS = plt.cm.Set1(np.linspace(0, 1, CHOSEN_K))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# --- Plot A: Alle Cluster zusammen ---
ax = axes[0]
for cl in range(CHOSEN_K):
    mask_cl = cluster_labels == cl
    mean_traj = np.nanmean(trajectories_clean[mask_cl], axis=0)
    std_traj = np.nanstd(trajectories_clean[mask_cl], axis=0)
    
    ax.plot(rel_years_range, mean_traj, marker='o', linewidth=2.5,
            color=CLUSTER_COLORS[cl], label=f'Cluster {cl} (n={np.sum(mask_cl):,})')
    
    lower = np.maximum(mean_traj - std_traj, 0)
    upper = np.minimum(mean_traj + std_traj, 100)
    ax.fill_between(rel_years_range, lower, upper, alpha=0.15, color=CLUSTER_COLORS[cl])

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Years Relative to Fire', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Mean Trajectories per Cluster', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

# --- Plot B: Einzelne Subplots ---
ax = axes[1]
ax.axis('off')

# Stattdessen: Feature-Vergleich als Heatmap
cluster_feature_means = np.zeros((CHOSEN_K, len(FEATURE_NAMES)))
for cl in range(CHOSEN_K):
    mask_cl = cluster_labels == cl
    cluster_feature_means[cl] = np.nanmean(feature_matrix_clean[mask_cl], axis=0)

df_cluster_features = pd.DataFrame(
    cluster_feature_means,
    index=[f'Cluster {cl}' for cl in range(CHOSEN_K)],
    columns=FEATURE_NAMES
)

ax2 = fig.add_subplot(122)
sns.heatmap(df_cluster_features, cmap='RdYlGn', annot=True, fmt='.1f',
            ax=ax2, linewidths=0.5, center=0,
            cbar_kws={'label': 'Mean Feature Value (unstandardized)'})
ax2.set_title('Cluster Feature Profiles', fontsize=13, fontweight='bold')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle(f'Mediterranean Trajectory Clusters (k={CHOSEN_K})',
             fontsize=15, fontweight='bold')
plt.tight_layout()
traj_plot_path = os.path.join(out_dir, "cluster_trajectories.png")
plt.savefig(traj_plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {traj_plot_path}")

# =====================================================
# 7b: Landcover-Zusammensetzung pro Cluster
# =====================================================

print("\n7b. Landcover-Zusammensetzung pro Cluster...")

LC_COLORS_CLUSTER = {
    1: '#228B22', 2: '#8B4513', 3: '#DAA520', 4: '#90EE90',
    5: '#4682B4', 6: '#FFD700', 7: '#808080', 8: '#D3D3D3', 9: '#0000FF'
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Stacked Bar: LC-Anteil pro Cluster ---
ax = axes[0]

lc_composition = {}
for cl in range(CHOSEN_K):
    mask_cl = cluster_labels == cl
    lc_cl = lc_valid[mask_cl]
    unique_lc_cl, counts_lc_cl = np.unique(lc_cl[lc_cl > 0], return_counts=True)
    total_cl = counts_lc_cl.sum()
    
    lc_composition[f'Cluster {cl}'] = {}
    for mid, cnt in zip(unique_lc_cl, counts_lc_cl):
        lc_name = MAJOR_LC_NAMES[mid]
        lc_composition[f'Cluster {cl}'][lc_name] = cnt / total_cl * 100

df_lc_comp = pd.DataFrame(lc_composition).T.fillna(0)

df_lc_comp.plot(kind='bar', stacked=True, ax=ax,
                color=[LC_COLORS_CLUSTER.get(
                    [k for k, v in enumerate(MAJOR_LC_NAMES) if v == col][0] if col in MAJOR_LC_NAMES else 0,
                    '#808080'
                ) for col in df_lc_comp.columns])

ax.set_ylabel('Percentage (%)')
ax.set_title('Land Cover Composition per Cluster', fontsize=12, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)
ax.grid(True, alpha=0.3, axis='y')

# --- Cluster-Anteil pro LC-Klasse ---
ax = axes[1]

lc_cluster_dist = {}
for mid in sorted(set(lc_valid[lc_valid > 0])):
    lc_name = MAJOR_LC_NAMES[mid]
    mask_lc = lc_valid == mid
    
    if np.sum(mask_lc) < 10:
        continue
    
    cl_of_lc = cluster_labels[mask_lc]
    unique_cl, counts_cl = np.unique(cl_of_lc, return_counts=True)
    total_lc = counts_cl.sum()
    
    lc_cluster_dist[lc_name] = {}
    for cl, cnt in zip(unique_cl, counts_cl):
        lc_cluster_dist[lc_name][f'Cluster {cl}'] = cnt / total_lc * 100

df_cl_dist = pd.DataFrame(lc_cluster_dist).T.fillna(0)

df_cl_dist.plot(kind='bar', stacked=True, ax=ax,
                color=[CLUSTER_COLORS[int(col.split()[-1])] for col in df_cl_dist.columns])

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution per Land Cover Type', fontsize=12, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Cluster × Land Cover Analysis – Mediterranean (k={CHOSEN_K})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
lc_plot_path = os.path.join(out_dir, "cluster_landcover_composition.png")
plt.savefig(lc_plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {lc_plot_path}")

# =====================================================
# 7c: Einzelne Trajektorie-Subplots pro Cluster
# =====================================================

print("\n7c. Einzelne Subplots pro Cluster...")

n_cols = min(3, CHOSEN_K)
n_rows = int(np.ceil(CHOSEN_K / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
if CHOSEN_K == 1:
    axes = np.array([axes])
axes = axes.flatten()

for cl in range(CHOSEN_K):
    ax = axes[cl]
    mask_cl = cluster_labels == cl
    
    mean_traj = np.nanmean(trajectories_clean[mask_cl], axis=0)
    std_traj = np.nanstd(trajectories_clean[mask_cl], axis=0)
    
    # Zeichne individuelle Trajektorien (Subsample)
    n_show = min(200, np.sum(mask_cl))
    rng = np.random.default_rng(42)
    show_idx = rng.choice(np.sum(mask_cl), n_show, replace=False)
    
    for idx in show_idx:
        ax.plot(rel_years_range, trajectories_clean[mask_cl][idx],
                color=CLUSTER_COLORS[cl], alpha=0.02, linewidth=0.5)
    
    ax.plot(rel_years_range, mean_traj, color='black', linewidth=3, label='Mean')
    
    lower = np.maximum(mean_traj - std_traj, 0)
    upper = np.minimum(mean_traj + std_traj, 100)
    ax.fill_between(rel_years_range, lower, upper, alpha=0.3, color='gray')
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_title(f'Cluster {cl} (n={np.sum(mask_cl):,})', fontsize=12, fontweight='bold',
                 color=CLUSTER_COLORS[cl])
    ax.set_xlabel('Years Relative to Fire')
    ax.set_ylabel('Woody Cover (%)')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    ax.set_ylim(0, 100)
    ax.legend(loc='upper right')

for i in range(CHOSEN_K, len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Individual Trajectories per Cluster – Mediterranean (k={CHOSEN_K})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
indiv_plot_path = os.path.join(out_dir, "cluster_individual_trajectories.png")
plt.savefig(indiv_plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {indiv_plot_path}")

In [ ]:
# === SCHRITT 8: RÄUMLICHE KARTE & EXPORT ===

print("\n8. RÄUMLICHE KARTE & EXPORT")
print("-" * 70)

# === 8a: Erstelle Cluster-Raster ===
cluster_raster = np.full((height, width), 255, dtype=np.uint8)  # 255 = NoData
cluster_raster[y_valid, x_valid] = cluster_labels.astype(np.uint8)

# === 8b: Speichere als GeoTIFF ===
cluster_meta = meta.copy()
cluster_meta.update({
    'count': 1,
    'dtype': 'uint8',
    'nodata': 255,
    'compress': 'lzw'
})

cluster_tif_path = os.path.join(out_dir, f"mediterranean_trajectory_clusters_k{CHOSEN_K}.tif")
with rasterio.open(cluster_tif_path, 'w', **cluster_meta) as dst:
    dst.write(cluster_raster, 1)
    dst.set_band_description(1, f'Trajectory_Cluster_k{CHOSEN_K}')

print(f"  ✓ {cluster_tif_path}")

# === 8c: Räumliche Visualisierung ===
from matplotlib.colors import ListedColormap, BoundaryNorm

fig, ax = plt.subplots(figsize=(16, 12))

# Erstelle Custom Colormap
cluster_cmap = ListedColormap([CLUSTER_COLORS[i] for i in range(CHOSEN_K)])
bounds = np.arange(-0.5, CHOSEN_K + 0.5, 1)
norm = BoundaryNorm(bounds, cluster_cmap.N)

# Maskiere NoData
cluster_display = np.ma.masked_where(cluster_raster == 255, cluster_raster)

# Berechne Extent aus Transform
extent = [
    transform[2],
    transform[2] + transform[0] * width,
    transform[5] + transform[4] * height,
    transform[5]
]

im = ax.imshow(cluster_display, cmap=cluster_cmap, norm=norm, extent=extent,
               interpolation='nearest')

cbar = plt.colorbar(im, ax=ax, ticks=range(CHOSEN_K), shrink=0.7)
cbar.set_ticklabels([f'Cluster {i}' for i in range(CHOSEN_K)])
cbar.set_label('Trajectory Cluster')

ax.set_title(f'Mediterranean Fire Trajectory Clusters (k={CHOSEN_K})\n'
             f'Single Fire Events, n={n_valid:,} Pixels',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

plt.tight_layout()
map_path = os.path.join(out_dir, f"cluster_map_mediterranean_k{CHOSEN_K}.png")
plt.savefig(map_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {map_path}")

# === 8d: Speichere Cluster-Summary CSV ===
summary_rows = []
for cl in range(CHOSEN_K):
    mask_cl = cluster_labels == cl
    n_cl = np.sum(mask_cl)
    
    row = {
        'Cluster': cl,
        'N_Pixels': int(n_cl),
        'Pct_of_Total': n_cl / n_valid * 100
    }
    
    # Feature-Mittelwerte
    for i, feat_name in enumerate(FEATURE_NAMES):
        row[f'{feat_name}_mean'] = np.nanmean(feature_matrix_clean[mask_cl, i])
        row[f'{feat_name}_median'] = np.nanmedian(feature_matrix_clean[mask_cl, i])
        row[f'{feat_name}_std'] = np.nanstd(feature_matrix_clean[mask_cl, i])
    
    # LC-Zusammensetzung
    lc_cl = lc_valid[mask_cl]
    for mid in range(1, 10):
        lc_name = MAJOR_LC_NAMES[mid]
        pct = np.sum(lc_cl == mid) / n_cl * 100 if n_cl > 0 else 0
        row[f'LC_{lc_name}_pct'] = pct
    
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
summary_csv = os.path.join(out_dir, f"cluster_summary_mediterranean_k{CHOSEN_K}.csv")
df_summary.to_csv(summary_csv, index=False)
print(f"  ✓ {summary_csv}")

# === 8e: Speichere Pixel-Level Zuordnung ===
pixel_rows = {
    'row': y_valid,
    'col': x_valid,
    'cluster': cluster_labels,
    'lc_major_id': lc_valid,
}
for i, feat_name in enumerate(FEATURE_NAMES):
    pixel_rows[feat_name] = feature_matrix_clean[:, i]

df_pixels = pd.DataFrame(pixel_rows)
pixel_csv = os.path.join(out_dir, f"pixel_cluster_assignment_mediterranean_k{CHOSEN_K}.csv")
df_pixels.to_csv(pixel_csv, index=False)
print(f"  ✓ {pixel_csv}")

# === ABSCHLUSSMELDUNG ===
print("\n" + "=" * 80)
print(f"✓✓✓ MEDITERRANEAN TRAJECTORY CLUSTERING ABGESCHLOSSEN (k={CHOSEN_K}) ✓✓✓")
print("=" * 80)
print(f"\n  📊 Pixel analysiert: {n_valid:,}")
print(f"  📊 Cluster: {CHOSEN_K}")
print(f"  📊 Features: {len(FEATURE_NAMES)}")
print(f"  📊 SOM-Größe: {som_side}×{som_side}")
print(f"\n  📈 PLOTS:")
print(f"     cluster_validation_metrics.png")
print(f"     cluster_trajectories.png")
print(f"     cluster_landcover_composition.png")
print(f"     cluster_individual_trajectories.png")
print(f"     cluster_map_mediterranean_k{CHOSEN_K}.png")
print(f"\n  📋 CSV/TIF:")
print(f"     cluster_validation_results.csv")
print(f"     cluster_summary_mediterranean_k{CHOSEN_K}.csv")
print(f"     pixel_cluster_assignment_mediterranean_k{CHOSEN_K}.csv")
print(f"     mediterranean_trajectory_clusters_k{CHOSEN_K}.tif")
print(f"\n  📁 Output: {out_dir}")
print("=" * 80)